# Sconto · cargar el dataset recolectado en Colab
Este notebook sólo carga y verifica datos; el análisis se desarrolla después.
Carga el último paquete público consistente de GitHub y verifica sus hashes. Para un snapshot fijo, cambia la URL por el ZIP fechado del release `datos`. También puedes usar `ORIGEN = 'subir'` para cargar un ZIP local. No requiere rutas de Windows, tokens ni conexión con las tiendas.
El paquete incluye precios históricos y capturas actuales realmente obtenidas. Los faltantes no se imputan y un precio tachado no prueba engaño.

In [ ]:
import hashlib, io, json, zipfile
import pandas as pd
import requests
ORIGEN = 'github'
URL = 'https://github.com/d1512pb/sconto-reale-o-falso/releases/download/datos/sconto_colab_actual.zip'
if ORIGEN == 'subir':
    from google.colab import files
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError('Sube un solo ZIP generado por sconto.exportar')
    contenido_zip = next(iter(subidos.values()))
else:
    respuesta = requests.get(URL, timeout=180)
    respuesta.raise_for_status()
    contenido_zip = respuesta.content

In [ ]:
tablas = {}
with zipfile.ZipFile(io.BytesIO(contenido_zip)) as paquete:
    manifiesto = json.loads(paquete.read('paquete.json'))
    for nombre, descripcion in manifiesto['tablas'].items():
        contenido = paquete.read(descripcion['archivo'])
        if hashlib.sha256(contenido).hexdigest() != descripcion['sha256']:
            raise ValueError(f'Archivo alterado: {nombre}')
        tabla = pd.read_parquet(io.BytesIO(contenido))
        if len(tabla) != descripcion['filas']:
            raise ValueError(f'Filas distintas al manifiesto: {nombre}')
        tablas[nombre] = tabla
observaciones = tablas['observaciones']
productos = tablas['productos']
eventos = tablas['eventos']
series_evento = tablas['series_evento']
print('Dataset verificado:', manifiesto['dataset_construido_utc'])
print({nombre: len(tabla) for nombre, tabla in tablas.items()})

Las variables anteriores pueden usarse en el notebook de análisis del equipo. Mantener separados vendedor/sucursal, canal, variante y moneda. Revisar `exclusiones` y `confianza_match` antes de comparar descuentos.
`pct_descuento_anunciado` es lo mostrado por la fuente. El porcentaje implícito calculado desde precio anterior se considera derivado, no una promesa textual. Las reglas con brecha 5/10/15 pp no son etiquetas verificadas de fraude.